# 🌟 Desafio Detecção de Anomalia no Dataset Pulsar

## 🎯 Objetivo

Neste desafio, você irá aplicar técnicas de **detecção de anomalias** utilizando o algoritmo **LOF (Local Outlier Factor)** no dataset de **estrelas pulsar**.

O objetivo é identificar **exemplos anômalos** no conjunto de dados, simulando um cenário onde não se conhece previamente quais objetos são estrelas pulsar e quais não são. Para isso, você utilizará o **LOF como método não supervisionado**, com foco na identificação de padrões incomuns.

Embora o dataset contenha a coluna `target_class` com os rótulos reais, **considere inicialmente que essa informação não está disponível para o modelo**. Ela poderá ser usada posteriormente para **avaliar a qualidade da detecção de anomalias**.

---

## 📊 Sobre o Dataset

O dataset contém medidas extraídas de observações astronômicas, com o intuito de classificar se uma determinada amostra corresponde ou não a uma estrela pulsar.

Cada linha representa uma observação, e as colunas contêm **estatísticas extraídas de séries temporais do sinal de rádio** de possíveis pulsares.

### 🔎 Colunas do Dataset

| Coluna                                       | Descrição                                                                 |
|----------------------------------------------|--------------------------------------------------------------------------|
| `Mean of the integrated profile`             | Média do perfil integrado do sinal.                                     |
| `Standard deviation of the integrated profile` | Desvio padrão do perfil integrado.                                    |
| `Excess kurtosis of the integrated profile`   | Curtose do perfil integrado.                                           |
| `Skewness of the integrated profile`          | Assimetria (skewness) do perfil integrado.                             |
| `Mean of the DM-SNR curve`                    | Média da curva DM-SNR (Signal-to-Noise Ratio).                         |
| `Standard deviation of the DM-SNR curve`      | Desvio padrão da curva DM-SNR.                                         |
| `Excess kurtosis of the DM-SNR curve`         | Curtose da curva DM-SNR.                                               |
| `Skewness of the DM-SNR curve`                | Assimetria da curva DM-SNR.                                            |
| `target_class`                                | Classe real do objeto: **1** indica estrela pulsar, **0** indica não-pulsar. Usada **apenas para avaliação**. |


# Importar bibliotecas

In [48]:
import numpy as np
import pandas as pd

from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import recall_score

import optuna

# Carregar os Dados

In [49]:
# carregar dados
df_churn = pd.read_csv('datasets/pulsar.csv')

In [50]:
# Visualizar a estrutura dos dados
df_churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17898 entries, 0 to 17897
Data columns (total 9 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Mean of the integrated profile                17898 non-null  float64
 1   Standard deviation of the integrated profile  17898 non-null  float64
 2   Excess kurtosis of the integrated profile     17898 non-null  float64
 3   Skewness of the integrated profile            17898 non-null  float64
 4   Mean of the DM-SNR curve                      17898 non-null  float64
 5   Standard deviation of the DM-SNR curve        17898 non-null  float64
 6   Excess kurtosis of the DM-SNR curve           17898 non-null  float64
 7   Skewness of the DM-SNR curve                  17898 non-null  float64
 8   target_class                                  17898 non-null  int64  
dtypes: float64(8), int64(1)
memory usage: 1.2 MB


In [51]:
# Visualizar as primeiras linhas
df_churn.head(10)

,Mean of the integrated profile,Standard deviation of the integrated profile,Excess kurtosis of the integrated profile,Skewness of the integrated profile,Mean of the DM-SNR curve,Standard deviation of the DM-SNR curve,Excess kurtosis of the DM-SNR curve,Skewness of the DM-SNR curve,target_class
0,140.562500,55.683782,-0.234571,-0.699648,3.199833,19.110426,7.975532,74.242225,0
1,102.507812,58.882430,0.465318,-0.515088,1.677258,14.860146,10.576487,127.393580,0
2,103.015625,39.341649,0.323328,1.051164,3.121237,21.744669,7.735822,63.171909,0
3,136.750000,57.178449,-0.068415,-0.636238,3.642977,20.959280,6.896499,53.593661,0
4,88.726562,40.672225,0.600866,1.123492,1.178930,11.468720,14.269573,252.567306,0
5,93.570312,46.698114,0.531905,0.416721,1.636288,14.545074,10.621748,131.394004,0
6,119.484375,48.765059,0.031460,-0.112168,0.999164,9.279612,19.206230,479.756567,0
7,130.382812,39.844056,-0.158323,0.389540,1.220736,14.378941,13.539456,198.236457,0
8,107.250000,52.627078,0.452688,0.170347,2.331940,14.486853,9.001004,107.972506,0
9,107.257812,39.496488,0.465882,1.162877,4.079431,24.980418,7.397080,57.784738,0


In [52]:
# Contar quantidade de pulsar e não pulsar
df_churn['target_class'].value_counts()

target_class
0    16259
1     1639
Name: count, dtype: int64

In [53]:
# Distribuicao percentual de pulsar e não pulsar
df_churn['target_class'].value_counts(normalize=True) * 100

target_class
0    90.842552
1     9.157448
Name: proportion, dtype: float64

# Preparação da Base para Algoritmo LOF

In [54]:
# Selecionando as colunas para o algoritmo
X = df_churn.drop(columns=['target_class'])
y = df_churn['target_class']

In [55]:
numeric_features = X.columns
numeric_features

Index(['Mean of the integrated profile',
       'Standard deviation of the integrated profile',
       'Excess kurtosis of the integrated profile',
       'Skewness of the integrated profile', 'Mean of the DM-SNR curve',
       'Standard deviation of the DM-SNR curve',
       'Excess kurtosis of the DM-SNR curve', 'Skewness of the DM-SNR curve'],
      dtype='object')

In [56]:
# Criar Transformers
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
    ]
)

In [57]:
# Transformar os dados
X_transformed = preprocessor.fit_transform(X)

In [58]:
X_transformed

array([[ 1.14931702,  1.33483164, -0.66956953, ..., -0.37062547,
        -0.07279778, -0.28743812],
       [-0.3341682 ,  1.80226533, -0.01178476, ..., -0.5889241 ,
         0.50442694,  0.21158145],
       [-0.31437216, -1.05332222, -0.14523256, ..., -0.23532816,
        -0.12599609, -0.39137346],
       ...,
       [ 0.3218423 ,  1.95621968, -0.2993338 , ...,  1.67156847,
        -1.28807874, -0.94133005],
       [ 0.13362759,  1.07450972, -0.26005007, ..., -0.66485697,
         0.37825656,  0.27584987],
       [-2.10576204,  5.73546965,  0.87267394, ...,  1.97154554,
        -2.19732744, -0.97105168]])

# Treinar o algoritmo LOF

In [59]:
# Converter y para mesma base do y_pred qaue será utilizado na otimização
y_true = y.map(lambda x: -1 if x == 0 else 1)

In [ ]:
# Iremos utilizar Optuna para fazer maximizacao do recall obtido na deteção de anomalias ao consideral Pulsares como anomalias
def lof_objective(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 10, 50)
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=0.1)
    y_pred = lof.fit_predict(X_transformed)
    
    return recall_score(y_true, y_pred)

In [61]:
search_space = {'n_neighbors': list(range(10, 51))}
sampler = optuna.samplers.GridSampler(search_space=search_space)
estudo_lof = optuna.create_study(direction='maximize', sampler=sampler)

[I 2025-07-09 13:51:36,986] A new study created in memory with name: no-name-32094f96-2459-4456-a9c7-145a265e15f2


In [62]:
# Executar o Optuna para otimizar os hiperparâmetros
estudo_lof.optimize(lof_objective, n_trials=len(range(10, 51)))

[I 2025-07-09 13:51:37,449] Trial 0 finished with value: 0.8261134838316047 and parameters: {'n_neighbors': 48}. Best is trial 0 with value: 0.8261134838316047.
[I 2025-07-09 13:51:37,698] Trial 1 finished with value: 0.8316046369737645 and parameters: {'n_neighbors': 14}. Best is trial 1 with value: 0.8316046369737645.
[I 2025-07-09 13:51:37,983] Trial 2 finished with value: 0.8352654057352044 and parameters: {'n_neighbors': 22}. Best is trial 2 with value: 0.8352654057352044.
[I 2025-07-09 13:51:38,309] Trial 3 finished with value: 0.8322147651006712 and parameters: {'n_neighbors': 27}. Best is trial 2 with value: 0.8352654057352044.
[I 2025-07-09 13:51:38,622] Trial 4 finished with value: 0.8340451494813911 and parameters: {'n_neighbors': 26}. Best is trial 2 with value: 0.8352654057352044.
[I 2025-07-09 13:51:38,933] Trial 5 finished with value: 0.8316046369737645 and parameters: {'n_neighbors': 24}. Best is trial 2 with value: 0.8352654057352044.
[I 2025-07-09 13:51:39,321] Trial 

## Exibindo melhor resultado

In [63]:
# Melhor configuração obtida pelo optuna
best_params = estudo_lof.best_params
best_params

{'n_neighbors': 19}

In [64]:
# Treinar algoritmo e já gerar as classificações de anomalia para cada registro (ponto de dados)
best_lof = LocalOutlierFactor(n_neighbors=best_params.get("n_neighbors"), contamination=0.1)
y_pred = best_lof.fit_predict(X_transformed)
y_pred

array([ 1, -1,  1, ...,  1,  1, -1])

In [65]:
# Mostrar valores preditos (anomalia ou não anomalia)
# No sklearn, o predict gera um valor = -1 (anomalia) e valor = 1 (pontos normais)
y_pred

array([ 1, -1,  1, ...,  1,  1, -1])

In [66]:
# Identificar anomalias
outliears = y_pred == -1
inliners = y_pred == 1

# Contar anomalias e os pontos normais
num_outliers = np.sum(outliears)
num_inliers = np.sum(inliners)

# Apresentar estatísticas
print(f"Anomalias detectadas: {num_outliers}")
print(f"Pontos normais: {num_inliers}")

Anomalias detectadas: 1790
Pontos normais: 16108


In [67]:
# Calcular Score com base no valor de y (churn real da base)
# Usar recall, pois o objetivo principal é maximizar o TPR (True Positive Rate)
recall_score(y_true, y_pred)

np.float64(0.8401464307504576)